# 1. Remove rows belonging to certain industries (make this clean)

In [ ]:
run_overwrite = True

import ibis
import pandas as pd
from utils.f_0_dirs import get_data_dirs

error_industries = []

dirs = get_data_dirs(segment="build")
con = ibis.duckdb.connect(str(dirs.db_path))

table_fame_derived = con.table("fame_derived")
table_fame_fixed = con.table("fame_fixed")
table_fame_yearly = con.table("fame_yearly")

# Helper function to prevent repeating the same counting block 3 times
def get_table_counts(derived, fixed, yearly, count_col_name="count"):
    return pd.DataFrame({
        "table": ["fame_derived", "fame_fixed", "fame_yearly"],
        count_col_name: [derived.count().execute(), fixed.count().execute(), yearly.count().execute()]
    })

count_before_df = get_table_counts(table_fame_derived, table_fame_fixed, table_fame_yearly, "count_before")
print(f"✅ Counts before filtering:\n{count_before_df.to_markdown(index=False)}\n")

# ==========================================
# 1. FILTERED (Firms IN error_industries)
# ==========================================
table_fame_derived_filtered = table_fame_derived.filter(table_fame_derived.industry_codes.isin(error_industries))

# SEMI-JOIN: Keeps rows in the left table that have a match in the right table.
# This eliminates the need for an inner_join + manual .select()
table_fame_fixed_filtered = table_fame_fixed.semi_join(table_fame_derived_filtered, "registered_number")
table_fame_yearly_filtered = table_fame_yearly.semi_join(table_fame_derived_filtered, "registered_number")

count_errors_df = get_table_counts(table_fame_derived_filtered, table_fame_fixed_filtered, table_fame_yearly_filtered, "count_after")
print(f"✅ Counts after filtering:\n{count_errors_df.to_markdown(index=False)}\n")

# ==========================================
# 2. EXCLUDED (Firms NOT IN error_industries)
# ==========================================
table_fame_derived_excluded = table_fame_derived.filter(~table_fame_derived.industry_codes.isin(error_industries))

# ANTI-JOIN: Keeps rows in the left table that DO NOT have a match in the right table.
# This replaces the cumbersome left_join + isnull() filter + select()
table_fame_fixed_excluded = table_fame_fixed.anti_join(table_fame_derived_filtered, "registered_number")
table_fame_yearly_excluded = table_fame_yearly.anti_join(table_fame_derived_filtered, "registered_number")

count_excluded_df = get_table_counts(table_fame_derived_excluded, table_fame_fixed_excluded, table_fame_yearly_excluded, "count_excluded")
print(f"✅ Counts after exclusion:\n{count_excluded_df.to_markdown(index=False)}\n")

# ==========================================
# 3. VERIFICATION
# ==========================================
# Use Pandas vectorized math to check all three tables simultaneously
expected_counts = count_before_df["count_before"] - count_errors_df["count_after"]
matches = count_excluded_df["count_excluded"] == expected_counts

if not matches.all():
    failed_tables = count_excluded_df.loc[~matches, "table"].tolist()
    raise ValueError(f"❌ Excluded counts do not match expected counts for: {', '.join(failed_tables)}")

for _, row in count_excluded_df.iterrows():
    t_name = row["table"]
    cb_val = int(count_before_df.loc[count_before_df['table'] == t_name, 'count_before'].sum())
    ca_val = int(count_errors_df.loc[count_errors_df['table'] == t_name, 'count_after'].sum())
    print(f"✅ Excluded counts match expected counts for table {t_name}: {row['count_excluded']:,} = "
          f"{cb_val:,} - {ca_val:,}")

# Output a sample of 10 rows from each of the filtered tables for verification
# Refactor the below: if any table has a zero error count then don't display
count_derived = count_errors_df.loc[count_errors_df['table'] == 'fame_derived', 'count_after'].sum()
if count_derived > 0:
    sample_frac_derived = 10 / count_derived
    display(table_fame_derived_filtered.sample(sample_frac_derived).execute())
count_fixed = count_errors_df.loc[count_errors_df['table'] == 'fame_fixed', 'count_after'].sum()
if count_fixed > 0:
    sample_frac_fixed = 10 / count_fixed
    display(table_fame_fixed_filtered.sample(sample_frac_fixed).execute())
count_yearly = count_errors_df.loc[count_errors_df['table'] == 'fame_yearly', 'count_after'].sum()
if count_yearly > 0:
    sample_frac_yearly = 10 / count_yearly
    display(table_fame_yearly_filtered.sample(sample_frac_yearly).execute())

# ==========================================
# 4. SAFE DATABASE OVERWRITE (ATOMIC SWAP)
# ==========================================
if run_overwrite:
    print("💾 Writing clean data to temporary tables and performing atomic swaps...")

    # 1. Update fame_derived
    # Check if the excluded count is zero
    if count_derived > 0:
        con.create_table("fame_derived_clean", table_fame_derived_excluded, overwrite=True)
        con.drop_table("fame_derived")
        con.create_table("fame_derived", con.table("fame_derived_clean"), overwrite=True)
        print("✅ Successfully updated fame_derived table.")
    else:
        print("⚠️ Excluded count for fame_derived is zero. Skipping update to fame_derived table.")

    # 2. Update fame_fixed
    if count_fixed > 0:
        con.create_table("fame_fixed_clean", table_fame_fixed_excluded, overwrite=True)
        con.drop_table("fame_fixed")
        con.create_table("fame_fixed", con.table("fame_fixed_clean"), overwrite=True)
        print("✅ Successfully updated fame_fixed table.")
    else:
        print("⚠️ Excluded count for fame_fixed is zero. Skipping update to fame_fixed table.")

    # 3. Update fame_yearly
    if count_yearly > 0:
        con.create_table("fame_yearly_clean", table_fame_yearly_excluded, overwrite=True)
        con.drop_table("fame_yearly")
        con.create_table("fame_yearly", con.table("fame_yearly_clean"), overwrite=True)
        print("✅ Successfully updated fame_yearly table.")
    else:
        print("⚠️ Excluded count for fame_yearly is zero. Skipping update to fame_yearly table.")

✅ Counts before filtering:
| table        |   count_before |
|:-------------|---------------:|
| fame_derived |        8771336 |
| fame_fixed   |        9283479 |
| fame_yearly  |       45609753 |

✅ Counts after filtering:
| table        |   count_after |
|:-------------|--------------:|
| fame_derived |             0 |
| fame_fixed   |             0 |
| fame_yearly  |             0 |

✅ Counts after exclusion:
| table        |   count_excluded |
|:-------------|-----------------:|
| fame_derived |          8771336 |
| fame_fixed   |          9283479 |
| fame_yearly  |         45609753 |

✅ Excluded counts match expected counts for table fame_derived: 8,771,336 = 8,771,336 - 0
✅ Excluded counts match expected counts for table fame_fixed: 9,283,479 = 9,283,479 - 0
✅ Excluded counts match expected counts for table fame_yearly: 45,609,753 = 45,609,753 - 0
💾 Writing clean data to temporary tables and performing atomic swaps...
⚠️ Excluded count for fame_derived is not zero. Skipping updat

# 2. Migrate: drop table fame_derived and merge into fame_fixed

In [ ]:
import ibis
from utils.f_0_dirs import get_data_dirs

data_dirs = get_data_dirs(segment="build")
con = ibis.duckdb.connect(str(data_dirs.db_path))

table_fame_fixed = con.table("fame_fixed")
table_fame_derived = con.table("fame_derived")

# Safely cast to a list to avoid Tuple + List TypeErrors
# 0. Rename the columns to plural before processing to avoid confusion
# 1. Isolate the columns we need and drop exact duplicates
cols_fame_fixed = list(table_fame_fixed.columns)
table_fame_derived = table_fame_derived.rename(
    industry_code = "industry_codes",
    file_code = "file_codes"
)
table_fame_derived_skinny = table_fame_derived.select(
    "registered_number", "industry_code", "file_code"
).distinct()

# 2. Collapse any remaining multiple rows for the same registered_number 
# into comma-separated strings to guarantee a 1-to-1 join
count_skinny = table_fame_derived_skinny.count().execute()
print(f"🔍 Collapsing {count_skinny:,} rows for the same registered_number into comma-separated strings...")
table_fame_derived_collapsed = table_fame_derived_skinny.group_by("registered_number").aggregate(
    industry_codes=table_fame_derived_skinny.industry_code.group_concat(","),
    file_codes=table_fame_derived_skinny.file_code.group_concat(",")
)
count_collapsed = table_fame_derived_collapsed.count().execute()
print(f"✅ Collapsed to {count_collapsed:,} unique registered_number rows from fame_derived.")

# 3. Perform the left join
# Because the right table now has strictly unique keys, this will NEVER fan out.
table_fame_merged = table_fame_fixed.left_join(table_fame_derived_collapsed, "registered_number")

# 4. Select the exact final schema to clean up any join artifacts
table_fame_merged_skinny = table_fame_merged.select(cols_fame_fixed + ["industry_codes", "file_codes"])

# --- VERIFICATION ---
num_rows = table_fame_merged_skinny.count().execute()
fixed_rows = table_fame_fixed.count().execute()

print(f"✅ Number of rows in fame_merged_skinny: {num_rows:,}")
print(f"✅ Number of rows in fame_fixed: {fixed_rows:,}")

if num_rows != fixed_rows:
    raise ValueError("❌ Row count mismatch! The number of rows in fame_merged_skinny does not match fame_fixed.")

print("✅ Schema of fame_merged_skinny:")
print(table_fame_merged_skinny.schema())

🔍 Collapsing 9,047,723 rows for the same registered_number into comma-separated strings...
✅ Collapsed to 8,114,908 unique registered_number rows.
✅ Number of rows in fame_merged_skinny: 8,323,347
✅ Number of rows in fame_fixed: 8,323,347
✅ Schema of fame_merged_skinny:
ibis.Schema {
  company_name                       string
  registered_number                  string
  ticker_symbol                      string
  ro_address                         string
  ro_address_line_1                  string
  ro_address_line_2                  string
  ro_address_line_3                  string
  ro_address_line_4                  string
  ro_address_line_5                  string
  ro_city                            string
  ro_county                          string
  ro_postcode                        string
  ro_full_postcode                   string
  ro_country                         string
  ro_latitude                        string
  ro_longitude                       string
  ro_nuts_r

In [3]:
# Safe overwrite
print("💾 Writing fame_merged_skinny to temporary table fame_fixed_clean...")
con.create_table("fame_fixed_clean", table_fame_merged_skinny, overwrite=True)
print("💾 Executing 4-step safe table overwrite procedure...")
con.drop_table("fame_fixed")
con.create_table("fame_fixed", con.table("fame_fixed_clean"), overwrite=True)
con.drop_table("fame_fixed_clean")
print("✅ Successfully updated fame_fixed table with merged and collapsed data.")

con.drop_table("fame_derived")
print("✅ Successfully dropped fame_derived table after merging into fame_fixed.")

💾 Executing 4-step safe table overwrite procedure...
✅ Successfully updated fame_fixed table with merged and collapsed data.
✅ Successfully dropped fame_derived table after merging into fame_fixed.
